# Gas-well decline analysis, step by step

This notebook walks through the whole project one step at a time. Run each cell with **Shift+Enter**, read the note above it, and check the output against what the note says you should see.

**What the project does:** it takes real monthly gas production from six Kansas wells, fits Arps decline curves to it, tests how well those curves forecast, and forecasts the wells that are still producing.

**Words used below**
- **Mcf**: thousand standard cubic feet of gas. **MMcf**: million standard cubic feet (1 MMcf = 1,000 Mcf).
- **Mcf/d**: gas rate in Mcf per day.
- **Decline curve analysis (DCA)**: fitting a smooth curve to a well's falling rate and extending it into the future.
- **Arps curves**: the standard decline curves (Arps, 1945): exponential, harmonic and hyperbolic.
- **EUR (estimated ultimate recovery)**: all the gas a well will ever produce = produced so far + remaining.

## 1. Get the code and the data

This downloads the project from GitHub into Colab. Run it once per session.

In [ ]:
import os
if not os.path.exists("gas-decline-analysis"):
    !git clone -q https://github.com/Lissha01/gas-decline-analysis
%cd gas-decline-analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gasdca import arps
from gasdca.data import load_monthly, load_wells, well_series
print("Ready")

## 2. Look at the data

Two files in the `data` folder:
- `wells.csv`: one row per well (name, field, county, operator, location).
- `kgs_gas_monthly.csv`: one row per well per month: the gas volume sold that month, in Mcf.

The data comes from the Kansas Geological Survey (KGS), which publishes production for every Kansas lease. All six are single-well leases that started producing after 2000.

In [ ]:
wells = load_wells()
wells[["well", "field", "county", "operator"]]

In [ ]:
df = load_monthly()
print(len(df), "well-months in total")
df.head()

## 3. From monthly volume to daily rate

Decline curves use a **rate** (Mcf/d), but KGS reports a **volume per month** (Mcf). So:

rate = monthly volume ÷ days in the month

**Worked example:** Logan 'X' 9 produced 3,423 Mcf in January 2010. January has 31 days, so the rate is 3,423 ÷ 31 = **110.4 Mcf/d**.

KGS does not report how many days the well actually produced. A month with a week of downtime therefore shows a low rate. Those months are flagged as *anomalous* and left out of the fits later.

In [ ]:
s = well_series(df, "1042372714")      # Logan 'X' 9
s[["gas_mcf", "days", "rate"]].head()

You should see 3423 Mcf, 31 days and a rate of about 110.4 Mcf/d in the first row.

In [ ]:
plt.figure(figsize=(8, 4))
plt.semilogy(s.index, s["rate"].where(s["rate"] > 0), ".")
plt.ylabel("Gas rate (Mcf/d)"); plt.title("Logan 'X' 9: monthly rate, log scale")
plt.show()

On a **log scale**, an exponential decline is a straight line. A curve that bends and flattens over time points to b > 0 (hyperbolic or harmonic).

## 4. The Arps equations

- **Exponential (b = 0):** q = qi × e^(−Di·t)
- **Hyperbolic (0 < b < 1):** q = qi ÷ (1 + b·Di·t)^(1/b)
- **Harmonic (b = 1):** q = qi ÷ (1 + Di·t)

Where:
- q is the rate at time t (Mcf/d)
- qi is the initial rate (Mcf/d)
- Di is the initial decline rate (per year here)
- b is the hyperbolic exponent, which sets how fast the decline itself slows down

**Worked example:** qi = 100 Mcf/d, Di = 0.5 per year, b = 0.5, after t = 1 year:

q = 100 ÷ (1 + 0.5 × 0.5 × 1)^(1/0.5) = 100 ÷ 1.25² = 100 ÷ 1.5625 = **64.0 Mcf/d**

The code works in days, so Di = 0.5 per year becomes 0.5 ÷ 365.25 per day.

In [ ]:
Di_per_day = 0.5 / 365.25
print(arps.rate(365.25, qi=100, Di=Di_per_day, b=0.5))

Expected output: 64.0. Try b = 0 (exponential) and b = 1 (harmonic) and compare.

## 5. Fit the three models to one well

`full_history_fits` does three things:
1. Starts at the peak month.
2. Stops at the first shut-in of 3 or more months.
3. Leaves out anomalous months, then fits each model by least squares on log(rate).

Log scale is used so that a 10% miss counts the same at 300 Mcf/d as at 10 Mcf/d.

In [ ]:
from gasdca.workflow import full_history_fits
fits, info = full_history_fits(s)
pd.DataFrame([{"model": m, "qi (Mcf/d)": round(f.qi, 1), "Di (per year)": round(f.Di * 365.25, 3),
               "b": round(f.b, 2), "first-year decline": f"{f.first_year_decline:.0%}",
               "misfit (RMS of ln q)": round(f.rmse_log, 3)} for m, f in fits.items()])

The **hyperbolic** model always fits the history at least as well as the other two, because it has an extra free parameter, b. Fitting the past well doesn't mean forecasting well, though. The next step tests that.

## 6. The blind test (hindcast)

Pretend you are standing 3 years after the peak:
1. Fit each model to those 3 years only.
2. Forecast the next 5 years.
3. Compare the forecast with what actually happened.

This is the honest test of a forecasting method.

In [ ]:
from gasdca.workflow import hindcast
h = hindcast(s)
print(f"Actual gas in the 5 test years: {h['actual_mcf']/1000:.1f} MMcf")
for m in ["exponential", "harmonic", "hyperbolic"]:
    print(f"{m:12s} predicted {h['pred_' + m]/1000:6.1f} MMcf   error {h['err_' + m]:+.0%}")

For Logan 'X' 9 you should see the exponential model **under-predict** by about 24%. Across all six wells it under-predicts every time, with a mean absolute error of 28%. Harmonic does best here at 12%. The full table is in `results/hindcast.csv`.

## 7. Forecast with uncertainty (P90 / P50 / P10)

For wells still producing, we fit the last 5 years and forecast to an economic limit of 1 Mcf/d or the end of 2050, whichever comes first.

A single fitted curve hides how uncertain the forecast is. So we build an **ensemble**: 200 slightly different versions of the history. Each one keeps the fitted trend but reshuffles the month-to-month noise. We refit each version and forecast again. The spread of the 200 answers gives:
- **P90** (low case): 90% of the ensemble forecasts more than this.
- **P50** (middle case).
- **P10** (high case): only 10% forecast more than this.

The same bootstrap was used in the blind test above. There, the P90 to P10 range contained the actual volume for only 3 of the 6 wells. Noise-based ranges like this **understate** the real uncertainty, because the bigger risk is picking the wrong model.

In [ ]:
from gasdca.workflow import forward_forecast
r = well_series(df, "1030420805")      # ROBBINS 2-27, still producing
f = forward_forecast(r, as_of=df["month"].max())
print(f"Produced to date: {f['cum_to_date']/1000:.1f} MMcf")
print(f"Remaining  P90 {f['rem_p90']/1000:.1f}   P50 {f['rem_p50']/1000:.1f}   P10 {f['rem_p10']/1000:.1f} MMcf")

## 8. Run the whole analysis

This runs every step for all six wells and saves the tables in `results/` and the charts in `figures/`.

In [ ]:
!python run_analysis.py

In [ ]:
from IPython.display import Image, display
for name in ["rate_fits", "hindcast", "hindcast_errors", "forecast"]:
    display(Image(f"figures/{name}.png"))

## 9. Your turn

Change one thing at a time and predict the result before you run it:

1. In `gasdca/workflow.py`, change `Q_LIMIT = 1.0` to `5.0`. Which well loses all of its remaining reserves, and why?
2. In the blind test, try `hindcast(s, train_months=24)` and then `train_months=60`. Does more history make the exponential model better or worse?
3. Change `DMIN_ANNUAL` from 0.06 to 0.10. Which forecast changes the most?
4. Explain in two sentences why the hyperbolic fit for CLIFT 'A' 3-33 has b = 1.84, and why that would be dangerous to extend for 30 years without a terminal decline.